# Proyecto Final Data Science II

**Clasificación reproducible de casos malignos y benignos**

Autor: Federico Hermann

Este notebook ejecuta el mismo pipeline canónico publicado en `src/project_pipeline.py`.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = "https://github.com/FedericoHermann/Data_Science_II_Hermann_Federico.git"
IN_COLAB = Path("/content").exists() and "google.colab" in sys.modules
if IN_COLAB:
    project = Path("/content/Data_Science_II_Hermann_Federico")
    if project.exists():
        subprocess.run(["git", "-C", str(project), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(project)], check=True)
else:
    candidates=[Path.cwd(), *Path.cwd().parents]
    project=next((p for p in candidates if (p/"src/project_pipeline.py").is_file()), None)
    if project is None:
        raise FileNotFoundError("No se encontró la raíz del repositorio")
os.chdir(project)
print("Proyecto:", Path.cwd())

In [ ]:
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
else:
    print("Entorno local: se utilizan las dependencias instaladas.")

## 1. Formulación del problema

- Tarea: clasificación binaria.
- Objetivo: `malignant`, con `1 = maligna`.
- Métrica principal: ROC-AUC.
- Costos previos a prueba: falso negativo = 5; falso positivo = 1.
- El conjunto de prueba permanece aislado hasta la evaluación final.

## 2. EDA e ingeniería de atributos

El pipeline valida datos, genera resúmenes, faltantes, outliers, distribuciones, boxplots y correlaciones. La ingeniería crea tres ratios dentro del pipeline para evitar fuga de información.

## 3. Entrenamiento, optimización y selección

Se comparan Logistic Regression, XGBoost baseline y XGBoost optimizado con GridSearchCV. La estimación se realiza con validación cruzada anidada. La selección usa una regla definida antes de prueba.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "py_compile", "src/project_pipeline.py"], check=True)
subprocess.run([sys.executable, "src/project_pipeline.py"], check=True)
print("Pipeline completo ejecutado correctamente.")

## 4. Resultados y criterio de decisión

In [ ]:
import json
import pandas as pd
from IPython.display import display, Markdown, Image
summary=json.loads(Path("artifacts/final_summary.json").read_text(encoding="utf-8"))
cv=pd.read_csv("reports/nested_cv_results.csv")
selection=pd.read_csv("reports/model_selection_table.csv")
test=pd.read_csv("reports/test_results.csv")
display(cv)
display(selection)
display(test)
display(Markdown(Path("reports/final_report.md").read_text(encoding="utf-8")))

## 5. Visualizaciones de evaluación

In [ ]:
for path in ["figures/model_comparison.png","figures/roc_pr_curves.png","figures/confusion_matrix.png"]:
    print(path); display(Image(filename=path))

## 6. Explicabilidad SHAP

SHAP se aplica al XGBoost optimizado como modelo explicativo complementario. Si el modelo seleccionado es distinto, estas explicaciones no se atribuyen al modelo final.

In [ ]:
for path in ["figures/shap_importance.png","figures/shap_beeswarm.png","figures/shap_local_explanation.png"]:
    print(path); display(Image(filename=path))
display(pd.read_csv("reports/shap_stability.csv").head(15))

## 7. Auditoría final

In [ ]:
subprocess.run([sys.executable, "scripts/audit_project.py"], check=True)
checks=json.loads(Path("artifacts/consistency_checks.json").read_text(encoding="utf-8"))
display(pd.DataFrame(list(checks.items()), columns=["control","resultado"]))
assert all(checks.values())
print("Todos los controles fueron superados.")

## 8. Conclusión

La recomendación final se genera desde los objetos de resultados, sin copiar números manualmente. El conjunto de prueba no reabre la selección. El proyecto es educativo y no constituye una herramienta clínica.